In [2]:
import pandas as pd 
import numpy as np 

In [5]:
from pathlib import Path


data_folder = Path(".")

In [6]:
performance_path=data_folder/"performance_reviews.csv"

In [7]:
performance=pd.read_csv(performance_path)

In [8]:
performance.shape

(10000, 4)

In [10]:
performance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   review_id    10000 non-null  int64  
 1   employee_id  9999 non-null   float64
 2   rating       8763 non-null   float64
 3   remarks      8407 non-null   object 
dtypes: float64(2), int64(1), object(1)
memory usage: 312.6+ KB


In [11]:
performance.describe()

,review_id,employee_id,rating
count,10000.00000,9999.000000,8763.000000
mean,5000.50000,5019.622962,3.103960
std,2886.89568,3168.706595,2.697575
min,1.00000,1.000000,-1.000000
25%,2500.75000,2540.000000,1.000000
50%,5000.50000,4977.000000,3.000000
75%,7500.25000,7473.500000,5.000000
max,10000.00000,99999.000000,8.000000


In [12]:
performance.head()

,review_id,employee_id,rating,remarks
0,1,1706.0,3.0,excellent
1,2,2178.0,5.0,Poor
2,3,6660.0,-1.0,NaN
3,4,2175.0,5.0,Good
4,5,4335.0,5.0,Poor


In [13]:
clean_performance=performance.copy()

In [15]:
clean_performance.columns=(
    clean_performance.columns
    .str.strip()
    .str.lower()
    .str.replace(r"_+","_",regex=True)
)

In [17]:
clean_performance=clean_performance.replace(
    r"^\s*$",
    np.nan,
    regex=True
)

In [18]:
clean_performance["review_id"].duplicated().sum()

0

In [19]:
clean_performance.duplicated().sum()

0

In [20]:
clean_performance["employee_id"] = pd.to_numeric(
    clean_performance["employee_id"],
    errors="coerce").astype("Int64")

In [21]:
employee_path= data_folder / "employees-cleaned.csv"

In [22]:
employee= pd.read_csv(employee_path)

In [23]:
invalid_employee_ids = clean_performance[
    ~clean_performance["employee_id"].isin(employee["employee__id"])
    & clean_performance["employee_id"].notna()
]
 
invalid_employee_ids

,review_id,employee_id,rating,remarks
7943,7944,99999,NaN,Poor
8962,8963,99999,NaN,Good


In [24]:
clean_performance= clean_performance[ clean_performance["employee_id"].isin(employee["employee__id"])].copy()

In [25]:
clean_performance["employee_id"].isna().sum()

0

In [26]:
clean_performance["rating"]= pd.to_numeric(
    clean_performance["rating"],errors="coerce"
)

In [27]:
clean_performance[(clean_performance["rating"]<1 ) | (clean_performance["rating"]>5)]

,review_id,employee_id,rating,remarks
2,3,6660,-1.0,NaN
5,6,7942,8.0,Excellent
12,13,4003,-1.0,Poor
13,14,2949,8.0,Poor
17,18,3353,-1.0,Good
...,...,...,...,...
9975,9976,8521,8.0,Average
9976,9977,5602,-1.0,Excellent
9979,9980,9897,8.0,excellent
9981,9982,797,-1.0,Excellent


In [29]:
clean_performance.loc[(clean_performance["rating"]<1 ) | (clean_performance["rating"]>5),"rating"]=np.nan

In [30]:
clean_performance["remarks"] = (
    clean_performance["remarks"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [31]:
clean_performance["remarks"] = clean_performance["remarks"].replace({
    "excellent":"Excellent",
    "good":"Good",
    "average":"Avergae",
    "poor":"Poor"
})

In [32]:
clean_performance["remarks"].value_counts(dropna=False)

remarks
Excellent    3392
Avergae      1694
Poor         1665
Good         1653
<NA>         1593
Name: count, dtype: Int64

In [35]:
clean_performance["remarks"]=(
    clean_performance["remarks"]
    .fillna("Not Provided")
)

In [36]:
clean_performance.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9997 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   review_id    9997 non-null   int64  
 1   employee_id  9997 non-null   Int64  
 2   rating       6238 non-null   float64
 3   remarks      9997 non-null   string 
dtypes: Int64(1), float64(1), int64(1), string(1)
memory usage: 400.3 KB


In [37]:
clean_performance.isna().sum()

review_id         0
employee_id       0
rating         3759
remarks           0
dtype: int64

In [39]:
clean_performance["rating"]=(
    clean_performance["rating"]
    .fillna("No Rating")
)

In [40]:
clean_performance.isna().sum()

review_id      0
employee_id    0
rating         0
remarks        0
dtype: int64

In [41]:
clean_performance.duplicated().sum()

0

In [45]:
clean_performance.to_csv(data_folder/"performance-cleaned.csv",index=False)